In [1]:
import os
import pandas as pd
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')
import re
import openpyxl
import numpy as np

In [ ]:
folder=r'Path'
OFolder=r'Output Folder Path'

In [ ]:
file_list=[]
for (root, dirs, file) in os.walk(folder):
    for f in file:
        if ('.xlsx') in f:

                    file_list.append(f)
file_list

In [4]:
file_link=[]

for i in range(len(file_list)):
     for r,d,f in os.walk(folder):
          for files in f:
               if files == file_list[i]:
                    file_link.append(os.path.join(r,files))

In [5]:
len(file_link)

66

In [ ]:
file_link[1]

In [7]:
cols=["Article Number",
"Brand",
"Region",
"Product Group",
"Product Group Overrides",
"Product Line",
"Publishing Countries",
"Article Status Code",
"Article Status Description",
"Children article numbers",
"Attribute",
"Value",
]
df_s = pd.DataFrame(columns=cols)
df_s


,Article Number,Brand,Region,Product Group,Product Group Overrides,Product Line,Publishing Countries,Article Status Code,Article Status Description,Children article numbers,Attribute,Value


In [ ]:
repl='''folder path\\'''
IDS=["Article Number",
"Brand",
"Region",
"Product Group",
"Product Group Overrides",
"Product Line",
"Publishing Countries",
"Article Status Code",
"Article Status Description"]

In [ ]:
for i in range(len(file_link)):
    workbook=openpyxl.load_workbook(file_link[i])
    print(file_link[i])
    df=pd.read_excel(file_link[i],sheet_name="article-export")
    df=pd.melt(df,id_vars=IDS,var_name="Attribute",value_name="Value").reset_index(drop=True)
    df_s=pd.concat([df_s,df])   


In [ ]:
import missingno as msno 
msno.matrix(df_s)

In [11]:
df_Match=df_s
df_Match.rename(columns={"Article Number":"PartNumber","Product Group":"PartTerminologyName",'Attribute':'PAName'},inplace=True)
df_Match=df_Match[["Brand",'PartTerminologyName',"PartNumber","PAName","Value"]]

In [ ]:
df_Match['Brand']="BPI"
df_Match

In [13]:
df_Match['PAName']=np.where(df_Match['PAName'].str.contains('(in)'), df_Match['PAName'].str.replace('(in)',""), df_Match['PAName'])
df_Match['PAName']=np.where(df_Match['PAName'].str.contains('(mm)'), df_Match['PAName'].str.replace('(mm)',""), df_Match['PAName'])
df_Match['PAName']=np.where(df_Match['PAName'].str.contains('(cm)'), df_Match['PAName'].str.replace('(cm)',""), df_Match['PAName'])

In [14]:
# unique_items_with_parentheses = df_Match[df_Match['PAName'].str.contains(r'\(', na=False)]['PAName'].unique()
# unique_items_with_parentheses

In [ ]:
df_Match

In [16]:
df_Attributes=df_Match[['Brand', 'PartTerminologyName',  'PAName']].drop_duplicates().reset_index(drop=True)

In [17]:
chunk_size=1000000
# Create a list of DataFrames by splitting the original DataFrame
df_chunks = [df_Match.iloc[i:i + chunk_size] for i in range(0, len(df_Match), chunk_size)]

In [18]:
with pd.ExcelWriter(OFolder+'\\'+'BPI_Attribute_List.xlsx') as writer:  # doctest: +SKIP
    for i, chunk in enumerate(df_chunks):
        sheet_name = f"Chunk_{i+1}"  # Naming each sheet dynamically
        chunk.to_excel(writer, sheet_name=sheet_name, index=False)
    df_Attributes.to_excel(writer,index=False, sheet_name='FBG_Attributes')

## Misc Codes

In [46]:
df_Final.to_csv('Combined_Data.csv', index=False)

In [47]:
df_articlemap=df_Final[['Article Number','Children article numbers']]
df_articlemap=df_articlemap.drop_duplicates().reset_index(drop=True)

In [48]:
df_articlemap['Children article numbers'] = df_articlemap['Children article numbers'].str.split(',')
df_articlemap=df_articlemap.explode('Children article numbers')
df_articlemap=df_articlemap.reset_index(drop=True)

In [49]:
df_articlemap.to_csv('BaseChildMap.csv', index=False)